# XGBoost

## 1. What is XGBoost?
XGBoost (eXtreme Gradient Boosting) is a scalable, open-source decision-tree ensemble built on gradient boosting (xgboost_1, xgb_2:37). It combines many shallow regression trees additively, each new tree correcting the errors of all previous ones, into a single strong model. The original paper's selling point is scalability: it handles sparse data, parallelizes tree building, and scales to billions of examples on one machine (bilions via out-of-core/compression/sharding), running ~10x faster than scikit-learn's GBM (xgboost_1:785). It won 17 of 29 Kaggle 2015 challenges.

## 2. Architecture
The model is a sum of K regression trees (CART), where each tree maps an input to a leaf and every leaf holds a continuous score (xgboost_1, Eq. 1). So the prediction is:
ŷᵢ = f₁(xᵢ) + f₂(xᵢ) + ... + f_K(xᵢ)
Key architectural pieces (xgboost_1, Sec. 2–4):
- **Regularized objective:** L = Σ loss(ŷᵢ, yᵢ) + Σ Ω(fₖ), where Ω(f) = γT + ½λ‖w‖² penalizes the number of leaves T and leaf weights w to stop overfitting (xgb_2 Eq. 6 calls γT a "pre-pruning" — higher γ → simpler trees). The loss part, Σ loss(ŷᵢ, yᵢ), simply measures "how wrong is the prediction ŷ vs. the true value y" — summed over all data points. Lower is better.

But if a model only cares about being right, it overfits. It memorizes the training data perfectly but fails on new data. So XGBoost adds a second term, Σ Ω(fₖ), a "complexity tax" on each tree:
Ω(f) = γT + ½λ‖w‖²
- T = number of leaves in the tree. Bigger T = more complicated tree. γ·T charges a fee per leaf, so the tree won't grow extra leaves unless they genuinely help.
- w = the numbers stored in those leaves (the actual predictions). ½λ‖w‖² charges a fee proportional to the size of those numbers — it nudges leaves toward small, moderate values instead of huge ones.

**Put simply:** XGBoost wants trees that are accurate but also simple — few leaves, modest values. The γT part is called "pre-pruning": if a split does not reduce error by at least γ, the tree just doesn't add that leaf (it prunes itself while growing, hence "pre"). Raise γ → fewer leaves → simpler, safer, shallower trees.
λ is the user set knob that controls how strong the penalty is.

- **Shrinkage:** each tree's contribution is scaled by learning rate η (like SGD learning rate). Boosting builds an ensemble one tree at a time, each new tree trying to fix what the previous ones got wrong. Shrinkage says: don't let any single tree have full effect.
Instead of adding a whole tree's prediction, only add a fraction of it:
new prediction = old prediction + η × (new tree's output)
with η typically 0.05 or 0.1 (your forecaster uses learning_rate=0.05, xgboost_forecaster.py:74).
Why? It's insurance against overfitting. If one tree is allowed to fully correct the errors right now, the model latches onto quirks of this particular training data too strongly. By only moving a small step each time, the model is forced to fix errors gradually across many trees — like SGD's learning rate, where you take small steps toward the minimum instead of one giant jump. The classic analogy: small steps = more stable, generalizes better; it just needs more steps (trees) to converge. That's why your forecaster uses a small learning_rate but a large n_estimators=300

- **Column and row subsampling:** 
    - **column subsampling:** when building each tree, don't consider all features at every split. Only look at a random subset of columns when picking a split. colsample_bytree=0.8 in your forecaster (xgboost_forecaster.py:144) = each tree only considers 80% of the lag/calendar features.
    - **row subsampling:** don't train each tree on all rows; give it a random 80% of them (subsample=0.8, xgboost_forecaster.py:143). Each tree sees a slightly different slice of history.
    - **Why it's done:**  if every tree sees all features and all rows, they all learn the same patterns and vote identically — the ensemble is just one big tree. By jittering features and rows per tree, each tree learns a different view, and their combined opinion (summed prediction) is smoother and generalizes better. This is exactly how Random Forest works (random features + bootstrap rows), which is why XGBoost borrows the idea

- **Pre-sorted column blocks:** Finding a good split means scanning a feature, checking "what if I split here?" at every value. Sorting is the expensive part. Normal libraries re-sort per node — wasteful. XGBoost sorts each feature once upfront and stores it as a ready column (= CSC format). From then on, split-finding is just a quick scan of the sorted column. Splitting is actually a sorting problem. For eg: for feature lag_48, if I cut at value 5000 vs 5001 vs ... — which single point separates lazy days from peak days best?" To evaluate a cut, you need to know the statistics (gradient sums) of points left of the cut vs right of the cut, for every possible cut. If data is in *unsorted* order, checking cut-at-value-5000 means scanning all rows (is each one ≤ 5000 or > 5000?). Checking the next candidate cut at 5001 means rescanning everything again. That's O(n) per candidate = catastrophically slow. If data is *sorted by the feature value*, you just walk down the sorted list once, adding each point to the "left side" as you pass it:

- **Parallel split-finding:** Since each feature column is independent, XGBoost searches for the best split in multiple columns at the same time on different CPU threads. That's why your forecaster's n_jobs=4 (xgboost_forecaster.py:145) makes training faster.

- **Sparsity-aware (handles missing values)** Real data has gaps (your demand series has NaN lags). XGBoost doesn't ignore them — at each node it *learns a "default direction"*: if the value is missing, send it left or right automatically, whichever worked best during training. No manual imputation needed.

## How does it work
XGBoost = train one tiny tree, learn from its mistakes, train another tree on those mistakes, repeat. Final answer = sum of all trees.

**Our mini-example:** predict today's demand y from one feature lag_48 (demand 24h earlier). 4 training rows:
x (lag_48)   y (demand)
4800         4900
5000         4950
5200         5300
5300         5350

**Step 0 — Start dumb:** Predict the average for everyone: mean = 5125. So all four forecasts are 5125, and the errors are:
actual  forecast  error (actual − forecast)
4900    5125      -225
4950    5125      -175
5300    5125      +175
5350    5125      +225

**Step 1 — Measure what needs fixing:** XGBoost turns each error into two numbers, the gradient g (direction/size of the mistake — roughly "negative error") and h (the second derivative, =1 for squared error). We just need g:
g values:  +225, +175, -175, -225
           (we were too high → need to come DOWN)
           (we were too low  → need to come UP)

**Step 2 — Grow a tree that fixes these:** Try a split: "x ≤ 5100" puts rows 1–2 on the left, rows 3–4 on the right.
Gain of the split = ½[GL²/(HL+λ) + GR²/(HR+λ) − G²/(H+λ)] − γ (xgboost_1, Eq. 7):
- Left: GL = 225+175 = 400, HL = 2 → 400²/2 = 80,000
- Right: GR = −175−225 = −400, HR = 2 → 80,000
- Parent: G = 0 → term is 0
- Gain = ½[80,000+80,000−0] − γ = 80,000 − γ
Huge gain → this split clearly separates "low-demand" rows (left) from "high-demand" rows (right). If the gain hadn't beaten γ, the tree would refuse to split (that's the pre-pruning from before).

**Step 3 — Decide how much to say.** Each leaf's value: w = −G/(H+λ):
- Left leaf: −400/2 = −200 → "if x ≤ 5100, subtract 200"
- Right leaf: −(−400)/2 = +200 → "if x > 5100, add 200"

**Step 4 — Shrink it.** Don't apply the full tree, only η = 0.1 × it. New predictions:
5125 + 0.1×(−200) = 5105   for rows 1–2   (moving 5125 → toward 4900/4950)
5125 + 0.1×(+200) = 5145   for rows 3–4   (moving 5125 → toward 5300/5350)
Small step — but notice the errors did shrink in the right direction.

**Step 5 — Repeat.** Recompute g from the new predictions (everyone got a bit closer), add another tree, shrink again. After 300 rounds, the 300 little corrections stack up and the sum lands close to the true values.

**Prediction time.** New row with x = 5150 falls into the right side of our first tree, right side of some trees, left of others (deeper splits) — it accumulates all 300 leaves' contributions, and the total is the forecast. No training, just routing each row down the trees and summing.

## 4. How it connects to the 4 adaptation arms
The contract is adapter(changepoints, model, data) -> updated model (adaptation/base.py:3), and Step 5 requires four arms, each a small Adapter subclass:
- **Arm 1 — never retrain:** XGBoost is fit once on the 2018–2019 window and rolled forward untouched. adapt() returns the model unchanged. Cheapest, but inert under COVID (2020-03) and 5MS cutover (2021-10) drift.
- **Arm 2 — retrain on a schedule:** adapt() refits XGBoost on a fixed cadence (e.g. every N days) regardless of changepoints — XGBoost's fit() is incremental-friendly here since our fit just rebuilds feature+target and calls xgb.XGBRegressor.fit, so retraining = calling model.fit(X, y) again.
- **Arm 3 — retrain on drift, full history:** when changepoints is non-empty, refit XGBoost on all data up to that point (data), i.e. full history including previous drift segments. The model re-learns lags/levels from scratch over everything.
- **Arm 4 — retrain on drift, recent window:** same trigger, but fit on only data.tail(window) (e.g. 60 days * 48 rows), as in adaptation/base.py:18-28. Drops old COVID-era patterns that no longer describe the new regime.

In [ ]:
# Implementation

import numpy as np
import pandas as pd
import xgboost as xgb

from drift_forecasting.forecasting.base import Forecaster

# It's the list of calendar features — information derived purely from the timestamp that gets fed into XGBoost as inputs (features) alongside the lag values.
# What each means:
# - hour / minute — time of day (so the model learns demand is higher at 7pm than 3am)
# - day_of_week — weekday vs weekend (weekend demand differs)
# - month — season (winter/summer demand differs)

_CALENDAR_COLUMNS = (
    "hour",
    "minute",
    "day_of_week",
    "month",
)
# A lag is a past value of the target series, offset back in time. It's how you turn a time series into something XGBoost can learn from.
# For SA1 demand at a point t (say Mar 1 2020, 10:00):
# - lag_1 = demand 30 min earlier (09:30)
# - lag_2 = demand 60 min earlier (09:00)
# - lag_48 = demand one day earlier (Mar 1 2020... wait — 48×30min = exactly yesterday 10:00)
# - extra_lags like 336 = one week earlier
# Also note that so lag_2 at 2018-01-01 01:00 = 2018-01-01 lag_1 at 00:30

class XGBoostForecaster(Forecaster):

    '''
    - interval = Spacing between readings. Converts to pd.Timedelta; lag_1 = 30 min before, lag_2 = 60 min..
    - max_lag = Number of contiguous lags. 48×30min = 1 day of history.
    - extra_lags = Extra lags: 2 days, 3 days, 7 days back (weekly pattern).
    - n_estimators = Number of boosting rounds (trees).
    - max_depth = Max tree depth — deeper = more complex patterns, more overfit.
    - learning_rate = Step size per tree. Lower = smaller corrections, needs more trees.
    - random_state = Seed for reproducible randomness.
    - early_stopping_rounds = Stop training early if error plateaus for 10 rounds. None disables it.
    - validation_fraction = validation_fraction is the slice of your 2018–2019 training data that the model doesn't learn from — it's set aside, like a practice exam, to check how well the model is doing. Note that validation is not same as testing. Training data is divided into 2 parts, training and validation. Validation is used to tune the model. Tuning means deciding when to stop training the model. The model trains in rounds. After each round, it checks how well it does on the validation set. If the validation score stops improving, it stops early instead of wasting time and potentially overfitting.
    '''    
    def __init__(
        self,
        interval: str = "30min",
        max_lag: int = 48,
        extra_lags: tuple[int, ...] = (96, 144, 336),
        n_estimators: int = 300,
        max_depth: int = 5,
        learning_rate: float = 0.05,
        random_state: int = 42,
        early_stopping_rounds: int | None = 10,
        validation_fraction: float = 0.15,
    ) -> None:
        if max_lag < 1:
            raise ValueError("max_lag must be >= 1")
        self.interval = interval
        self.max_lag = max_lag
        self.extra_lags = extra_lags
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.early_stopping_rounds = early_stopping_rounds
        self.validation_fraction = validation_fraction

        # convert interval to timedelta object
        self._interval = pd.Timedelta(interval)
        #the list of lag distances. range(1, 49) gives lags 1–48 (contiguous half-hours back to 24h), then | set((96, 144, 336)) unions in the extra sparse lags (48h, 72h, 1 week). sorted(set(...)) dedupes and orders them. Result: a power-set of "how far back to look."
        self._lags = sorted(set(range(1, max_lag + 1)) | set(extra_lags))
        #_feature_columns is just a list of names naming every input the model is allowed to look at when making a prediction — like the column headers of a spreadsheet.
        # For each lag distance (1, 2, 3, …, 48, 96, 144, 336), there's one column lag_k = "the demand value k half-hours ago." Plus 4 calendar columns.
        # So it looks like:
        # lag_1   lag_2  ...  lag_48  lag_96  lag_144  lag_336  hour  minute  day_of_week  month
        self._feature_columns = [f"lag_{lag}" for lag in self._lags] + list(
            _CALENDAR_COLUMNS
        )
        #- _history — A memory of past target values. When fit() runs, it saves the training data here. During predict(), it looks back at this history to fill in lag values (what happened 30 min ago, 1 hour ago, etc.). Each new forecast gets added to this history so the next forecast can use it.
        self._history: pd.Series | None = None
        #- _target_name — Remembers the name of the column being predicted (e.g., "TOTALDEMAND"). This is used during predict() to check if the input data X contains actual observed values for that column, so it can use real values instead of forecasts when available.
        self._target_name: str | None = None
        #  self._model — a storage slot on the object that will hold the actual trained tree model.
        self._model: xgb.XGBRegressor | None = None

    #fit() does two things:
    #1. Saves the training data — It stores the target values (y) and their timestamps in self._history so predict() can look them up later.
    #2. Trains the XGBoost model — It builds lag features (e.g., value 30 min ago, 1 hour ago, 2 days ago) and calendar features (hour, minute, day of week, month) from the training data, then trains a gradient-boosted tree regressor on those features to predict the target.
    #Parameters (inputs):
    #- X — A DataFrame of timestamps (the index) representing the training time periods
    #- y — A Series of target values (e.g., electricity demand) for each timestamp in X
    #Output:
    # - Returns self (the fitted forecaster object) — so you can chain calls like forecaster.fit(X, y).predict(X_test)
#     Side effects (what it changes inside):
#     - Populates self._history with the training values
#     - Populates self._target_name with the column name
#     - Populates self._model with the trained XGBoost model
    def fit(
        self,
        X: pd.DataFrame,
        y: pd.Series,
    ) -> "XGBoostForecaster":
        """Fit on the training window only. Returns self."""
        # Converts X.index to datetime — Makes sure the timestamps are proper datetime objects. X.index contains timestamps eg:  2018-01-01 00:00, 2018-01-01 00:30, 2018-01-01 01:00
        timestamps = self._as_datetime_index(X.index)

        # Aligns y to those timestamps — If y is missing any timestamps, they become NaN.
        y = y.reindex(timestamps)

        # Finds valid rows — keep is a boolean mask: True where y has a real value, False where it's NaN.
        keep = y.notna()

        # Saves history — Stores only the valid (non-NaN) values and timestamps in self._history.
        # Sample output: 
        # 2018-01-02 00:00    550.2
        # 2018-01-02 01:00    560.1
        # 2018-01-02 01:30    570.5
        # Name: TOTALDEMAND, dtype: float64
        self._history = pd.Series(
            y[keep].to_numpy(),
            index=timestamps[keep],
            # y.name is the column name of the Series y.
            name=y.name,
        )

        # Saves column name — Remembers the name of y (e.g., "TOTALDEMAND") in self._target_name.
        self._target_name = y.name

        # It builds a DataFrame with two types of columns:
        # 1. Lag columns — For each lag (1, 2, ..., 48, 96, 144, 336), it looks back in y and grabs the value from that many intervals ago.
        # 2. Calendar columns — Hour, minute, day of week, month from each timestamp.
        # The below is a sample table of how data looks like
        # timestamp lag_1 lag_2 lag_3 ... lag_48 lag_96 lag_144 lag_336 hour minute day_of_week month
        # 2019-01-02 00:00 550.2 540.1 530.8 ... 520.0 510.3 505.7 490.1 0 0 1 1
        # 2019-01-02 00:30 560.1 550.2 540.1 ... 530.5 520.8 515.2 500.4 0 30 1 1

        # Note that the data of 2018-01-01 00:00 (starting point) will be like this:
        # timestamp lag_1 lag_2 ... 
        # 2018-01-02 00:00 NaN NaN
        # 2018-01-02 00:30 value NaN
        # 2018-01-02 01:00 value value
        # ... ... ...
        # this is cuz data lag_1 (30 mins before) at 2018-01-02 00:00 doesn't exist as the data itself starts from there

        features = self._build_training_features(
            timestamps[keep],
            y[keep],
        )

        # Extracts targets — Grabs just the valid y values as a numpy array to train on.
        targets = y[keep].to_numpy()

        # Total training rows
        n = len(features)

        # The number of rows meant for validating
        n_validation = int(n * self.validation_fraction)

        # We can validate only if following conditions are met
        can_validate = (
            # Early stopping is enabled
            self.early_stopping_rounds is not None
            and self.early_stopping_rounds > 0
            # validation rows must atleast be 20. 20 rows is a minimum threshold to get a stable enough signal of whether the model is actually improving or not.
            and n_validation >= 20
            # Training data must be 2x bigger than validation data
            and (n - n_validation) >= n_validation
        )

        # model params
        params = {
            "n_estimators": self.n_estimators,
            "max_depth": self.max_depth,
            "learning_rate": self.learning_rate,
            "random_state": self.random_state,
            # Objective is to minimize squared error
            "objective": "reg:squarederror",
            # Evaluate performance using rmse
            "eval_metric": "rmse",
            # Each tree trains on a random 80% of rows (reduces overfitting)
            "subsample": 0.8,
            # Each tree uses a random 80% of features (reduces overfitting)
            "colsample_bytree": 0.8,
            # Use 4 CPU cores for training (speeds things up)
            "n_jobs": 4,
        }

        # If validation is possible, it adds a callback that tells XGBoost: "If the validation score hasn't improved for 10 rounds, stop training early."
        if can_validate:
            params["callbacks"] = [
                xgb.callback.EarlyStopping(rounds=self.early_stopping_rounds)
            ]

        # It creates an XGBoost regressor model using all the parameters defined above. **params unpacks the dictionary, so it becomes xgb.XGBRegressor(n_estimators=300,max_depth=5,learning_rate=0.05,...)
        self._model = xgb.XGBRegressor(**params)

        # If validation is possible:
        if can_validate:
            # Splits the data: first 85% for training, last 15% for validation
            split = n - n_validation
            self._model.fit(
                # Feature training data (From x)
                features.iloc[:split],
                # Target training data (From y)
                targets[:split],
                # In eval_set we define our validation data
                eval_set=[
                    (
                        features.iloc[split:],
                        targets[split:],
                    )
                ],
                # Print output during training
                verbose=True,
            )
        # If validation is not possible:
        else:
            # Uses all the data for training, no validation
            self._model.fit(features, targets, verbose=True)

        return self

    #Input X: a DataFrame indexed by datetime (the future steps to forecast). The only column it looks at is one named like the target y (optional — if present with non-NaN values, those take precedence for lag filling in walk-forward).
    # Output: a NumPy array with one forecasted value per row, in the same order as X.index.
    # It works by, for each timestamp:
    # 1. Building a feature row of lag_1 ... lag_336 (values from 30min up to 7 days back) + calendar features (hour, minute, day_of_week, month), from stored history or its own earlier forecasts (xgboost_forecaster.py:203-218).
    # 2. Passing that row to the trained XGBoost model → one forecast (xgboost_forecaster.py:220).
    # 3. Storing that forecast so later steps can use it as a lag (recursive multi-step, xgboost_forecaster.py:224-225).

    # Sample input: X = pd.DataFrame(index=[
    #     "2020-03-19 00:00",
    #     "2020-03-19 00:30",
    #     "2020-03-19 01:00",
    # ])
    # Sample output: array([5120.4, 4901.8, 4612.3]) forecasts for each timestamp

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Point forecasts, one per row of X, produced chronologically."""

        if self._model is None or self._history is None:
            raise RuntimeError("predict() called before fit()")

        original_order = self._as_datetime_index(X.index)

        timestamps = original_order.sort_values()

        actuals = None
        if self._target_name is not None and self._target_name in X.columns:
            actuals = dict(
                zip(
                    original_order,
                    X[self._target_name].to_numpy(),
                )
            )

        values = self._history.copy(deep=False)

        forecasts = {}

        feature_row = np.empty(
            (1, len(self._feature_columns)),
            dtype=np.float64,
        )

        columns = {name: index for index, name in enumerate(self._feature_columns)}

        for timestamp in timestamps:
            for lag in self._lags:
                lag_time = timestamp - lag * self._interval

                value = self._value_at(
                    lag_time,
                    actuals,
                    values,
                )

                feature_row[0, columns[f"lag_{lag}"]] = value

            feature_row[0, columns["hour"]] = timestamp.hour
            feature_row[0, columns["minute"]] = timestamp.minute
            feature_row[0, columns["day_of_week"]] = timestamp.dayofweek
            feature_row[0, columns["month"]] = timestamp.month

            forecast = self._model.predict(feature_row)[0]

            forecasts[timestamp] = forecast

            if timestamp not in values.index:
                values.loc[timestamp] = forecast

        return np.array([forecasts[timestamp] for timestamp in original_order])

    # It builds the feature matrix (rows = timestamps, columns = features) used to train the model.
    # It creates two sets of columns:
    # 1. Lag columns — For each lag in self._lags (1, 2, 3, ..., 48, 96, 144, 336):
    # - Looks back that many intervals in y
    # - Column name: lag_{lag} (e.g., lag_1, lag_96)
    # 2. Calendar columns — From each timestamp:
    # - hour, minute, day_of_week, month
    # Then combines them side-by-side into one DataFrame and returns it.
    # In short: It turns past values + time-of-day info into a table XGBoost can learn patterns from, where each row predicts the target at that timestamp.

    # Params:
    # - timestamps — The datetime labels (the index) telling when each observation happened. Used to build calendar features and to look up past values. It is a DatetimeIndex which is like a collection of timestamps
    # - y — The actual target values (e.g., electricity demand) in a Series. Used to fill each lag column by shifting back in time.

    # Sample input
    # - timestamps = ["2018-01-02 01:00", "2018-01-02 01:30"]
    # - y = {2018-01-02 00:00: 550.2, 00:30: 560.1, 01:00: 570.5, 01:30: 575.0}

    # Sample output:
    # Output:
    # timestamp	lag_1	lag_2	hour	minute	day_of_week	month
    # 2018-01-02 01:00	560.1	550.2	1	0	1	1
    # 2018-01-02 01:30	570.5	560.1	1	30	1	1

    def _build_training_features(
        self,
        timestamps: pd.DatetimeIndex,
        y: pd.Series,
    ) -> pd.DataFrame:
        """Lag + calendar features for the training window."""

        # It creates an empty DataFrame whose index is the timestamps and no columns yet.
        lags = pd.DataFrame(index=timestamps)

        # It loops through each lag value and fills in a lag column:
        for lag in self._lags:
            # offset — How far back in time, e.g., lag 2 × 30 min = 60 minutes
            offset = lag * self._interval

            # timestamps - offset — Returns the past timestamps (each shifted back by the offset) as a collection
            # reindex function in a series is used to change the order and add add a new index into the series
            # y.reindex(timestamps - offset) returns a collection of values corresponding to their timestamps. The value is nan if there is no existing value to the timestamp. to_numpy converts the collection of timestamps to array of timestamps
            # the array of timestamps is then assigned to lags dataframe as values.

            # For eg:
            # initially consider lags with index = [01:00, 01:30, 02:00], y={01:00: 570.5, 01:30: 575.0, 02:00: 580.0} and self._lags=[1,2]
            # Now lag=1, offset=1x30=30
            # y.reindex(timestamps-offset) = {(01:00 - 30): <value at (01:00 - 30)>, (01:30 - 30): <value at (01:30 - 30)>, (02:00 - 30): <value at (02:00 - 30)>} = {00:30: nan, 01:00: 570.5, 01:30: 575.0}
            # y.reindex(timestamps-offset).to_numpy() = [nan, 570.5, 575.0]
            # therefore lags["lag_1"]=[nan, 570.5, 575.0]
            # and the loop goes on for every offset
            lags[f"lag_{lag}"] = y.reindex(timestamps - offset).to_numpy()

        # It builds a DataFrame with four calendar columns extracted from each timestamp:
        # For eg: timestamps=["2018-01-02 01:30", "2018-01-02 02:00"]
        # calendar:
        # index,hour,minute,day_of_week,month
        # 2018-01-02 01:30,1,30,1,1
        # 2018-01-02 02:00,2,0,1,1
        calendar = pd.DataFrame(
            {
                "hour": timestamps.hour,
                "minute": timestamps.minute,
                "day_of_week": timestamps.dayofweek,
                "month": timestamps.month,
            },
            index=timestamps,
        )

        return pd.concat([lags, calendar], axis=1)

    # It looks up the value at a past timestamp (lag_time), in a specific priority order:
    # 1. Real observed values (values that is present in the test set) first — If the input data X contains an actual measured value at that time, use it
    # 2. Otherwise use history/forecasts (values predicted by the model) — Check values (training history + earlier forecasts) for that timestamp
    # 3. Fallback — If neither has it, return NaN
    # Why the priority? Real observed values (walk-forward testing) are more accurate than forecasts, so they're preferred. But when predicting the future, only forecasts exist, so those are used instead.

    # Params:
    # - actuals — A dictionary of real measured values from the test set input X. Built in predict() (line 183–190) by reading the column matching _target_name from X. Format: {timestamp: observed_value}. Can be None if X has no such column. But in walk forward evaluation, actuals won't be None as we already have the whole testing data.
    # - values — A pandas Series of the model's available data. Starts as a copy of _history (training values) and grows as each new forecast is appended during predict(). Format: index=timestamps, values=target.

    # Sample input:
    # - lag_time = t5
    # - actuals = {t5: 560.1} (real observed value at t5)
    # - values = pd.Series([550.2, 555.0], index=[t3, t5]) (history + prior forecasts)

    # Sample output:
    # Example 1:
    #  function call = c_value_at(t5, actuals, values)
    # actuals has t5=560.1, so returns 560.1  (ignoring 555.0 in values)
    # Example 2 — No real value, but history has it:
    # _value_at(t3, actuals, values)
    # t3 not in actuals, but t3=550.2 in values → returns 550.2
    # Example 3 — Nobody has it:
    # _value_at(t7, actuals, values)
    # t7 in neither → returns NaN

    @staticmethod
    def _value_at(
        lag_time: pd.Timestamp,
        actuals: dict | None,
        values: pd.Series,
    ) -> float:
        """Value at `lag_time`, preferring observed to forecast values."""

        # if actuals is not none then get the forecast. If the forecast is not none and nan then return it after converting it to python float from numpy float (np.float)
        if actuals is not None:
            actual = actuals.get(lag_time)
            if actual is not None and not pd.isna(actual):
                return float(actual)

        # if actuals is None then return it from values
        if lag_time in values.index:
            return values.loc[lag_time]

        # if not present in both then return nan
        return np.nan

    # It converts the input index to a DatetimeIndex — and raises an error if it can't.
    # Input (valid):
    #     index = ["2020-03-01 00:00", "2020-03-01 00:30", "2020-03-01 01:00"]
    # Output:
    #     DatetimeIndex(['2020-03-01 00:00:00', '2020-03-01 00:30:00', '2020-03-01 01:00:00'], dtype='datetime64[ns]')
    @staticmethod
    def _as_datetime_index(index) -> pd.DatetimeIndex:
        numeric = pd.api.types.is_numeric_dtype(pd.Series(index))

        if numeric:
            raise ValueError(
                "An X index of datetime values is required to build lag features"
            )

        try:
            return pd.DatetimeIndex(index)
        except (TypeError, ValueError) as error:
            raise ValueError(
                "An X index of datetime values is required to build lag features"
            ) from error

# References
1. Chen, T. and Guestrin, C. (2016). XGBoost: a Scalable Tree Boosting System. Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining - KDD ’16, 1(1), pp.785–794. doi:10.1145/2939672.2939785.
2. Bentéjac, C., Csörgő, A. and Martínez-Muñoz, G. (2020). A Comparative Analysis of Gradient Boosting Algorithms. Artificial Intelligence Review, [online] 54(3). doi:10.1007/s10462-020-09896-5.